# Kaggle ResNet18 SimCLR Runner

Use this notebook on Kaggle GPU for one ResNet18 SimCLR experiment at a time. The training logic stays in the repository scripts. This notebook only prepares Kaggle paths, verifies inputs, runs pretraining, then runs supervised fine-tuning.

## 1. Enable GPU

In Kaggle, open notebook settings and set Accelerator to GPU. Do not use TPU for this pipeline.

In [12]:
from pathlib import Path
import os
import shutil
import pandas as pd

print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())
!nvidia-smi

Kaggle input exists: True
Kaggle working exists: True
Sun May 31 08:50:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  

## 2. Clone or Pull Repository

In [13]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('REPO_ROOT:', REPO_ROOT)
!git rev-parse --short HEAD

/kaggle/working/contrastive-synthesis-medcls_CVProject
Already up to date.
REPO_ROOT: /kaggle/working/contrastive-synthesis-medcls_CVProject
09bbb255


## 3. Install Minimal Dependencies

Kaggle already includes PyTorch. Install only lightweight packages used by the scripts.

In [14]:
!pip install -q timm scikit-learn matplotlib pandas Pillow

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA available: True


## 4. Edit Kaggle Dataset Paths

After adding your Kaggle Dataset to this notebook, edit these paths to match the folder names under `/kaggle/input`. The helper cell below links them into the repository as `data/processed/...`, so the existing scripts do not need path changes.

In [15]:
from pathlib import Path
import os

DATA_ROOT = Path("/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data")

LABELLED_SOURCE = DATA_ROOT / "processed/labelled_4232"
UNLABELLED_SOURCE = DATA_ROOT / "processed/unlabelled_16934"
SYNTHETIC_SOURCE = DATA_ROOT / "processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan"
MANIFEST_SOURCE = DATA_ROOT / "manifests"

OUTPUT_ROOT = Path("/kaggle/working/results/experiments")
PREVIOUS_EXPERIMENT_SOURCE = None

EXPERIMENT_ID = "resnet18_covidqu_syn"

PRETRAIN_EPOCHS = 70
FINETUNE_EPOCHS = None

print("LABELLED_SOURCE:", LABELLED_SOURCE, LABELLED_SOURCE.exists())
print("UNLABELLED_SOURCE:", UNLABELLED_SOURCE, UNLABELLED_SOURCE.exists())
print("SYNTHETIC_SOURCE:", SYNTHETIC_SOURCE, SYNTHETIC_SOURCE.exists())
print("MANIFEST_SOURCE:", MANIFEST_SOURCE, MANIFEST_SOURCE.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

LABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232 True
UNLABELLED_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934 True
SYNTHETIC_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan True
MANIFEST_SOURCE: /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/manifests True
OUTPUT_ROOT: /kaggle/working/results/experiments
EXPERIMENT_ID: resnet18_covidqu_syn


## 5. Link Data and Prepare Manifests

In [16]:
import os

src = "/kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan"
dst = "/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan"

os.makedirs(os.path.dirname(dst), exist_ok=True)

if not os.path.exists(dst):
    os.symlink(src, dst)

print("exists:", os.path.exists(dst))
print("classes:", os.listdir(dst))

exists: True
classes: ['Viral_Pneumonia', 'Normal', 'Lung_Opacity', 'COVID']


In [17]:
import glob, os

img_dir = "/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images"
print(os.path.exists(img_dir))
print(len(glob.glob(img_dir + "/*.png")))
print(glob.glob(img_dir + "/*.png")[:3])

True
1000
['/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00479.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00134.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00619.png']


In [18]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        print('WARNING: source missing:', source)

%cd {REPO_ROOT}
replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/unlabelled_16934', UNLABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_SOURCE)

manifest_dir = REPO_ROOT / 'data/manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = manifest_dir / name
    if src.exists():
        shutil.copy2(src, dst)
        print('Copied manifest:', dst)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

# Normalize synthetic manifest for Kaggle. Stage 1 manifests made on Colab may contain Drive absolute paths.
src_syn_manifest = MANIFEST_SOURCE / 'synthetic_dcgan.csv'
dst_syn_manifest = manifest_dir / 'synthetic_dcgan.csv'
if src_syn_manifest.exists():
    df = pd.read_csv(src_syn_manifest)
    normalized_paths = []
    for _, row in df.iterrows():
        original = Path(str(row['image_path']))
        class_name = row['class_name']
        filename = original.name
        candidates = [
            Path('data/processed/synthetic_dcgan') / class_name / 'images' / filename,
            Path('data/processed/synthetic_dcgan') / class_name / filename,
            Path('data/processed/synthetic_dcgan') / original.name,
        ]
        selected = candidates[0]
        for candidate in candidates:
            if (REPO_ROOT / candidate).exists():
                selected = candidate
                break
        normalized_paths.append(str(selected))
    df['image_path'] = normalized_paths
    df.to_csv(dst_syn_manifest, index=False)
    print('Wrote Kaggle-normalized synthetic manifest:', dst_syn_manifest)
elif dst_syn_manifest.exists():
    print('Using repo synthetic manifest:', dst_syn_manifest)
else:
    print('WARNING: synthetic_dcgan.csv not found. Synthetic experiments will fail until this is provided.')

!find data -maxdepth 3 -type d | sort | head -40
!ls -lh data/manifests

/kaggle/working/contrastive-synthesis-medcls_CVProject
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/labelled_4232 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/labelled_4232
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/unlabelled_16934 -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/unlabelled_16934
Linked /kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan -> /kaggle/input/datasets/tlinhevg05/medcls-cvproject/data/processed/synthetic_dcgan-20260530T212146Z-3-001/synthetic_dcgan
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/train.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/val.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests/test.csv
Copied manifest: /kaggle/working/contrastive-synthesis-medcls_CVProject/data/manifests

## 6. Verify Inputs and Scripts

In [19]:
SYNTHETIC_MANIFEST = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'

!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_simclr_resnet.py scripts/run_classification_resnet.py
!python scripts/run_simclr_resnet.py --help | grep resume || true


Experiment Input Check Report
[PASS] common config
  - loaded configs/experiments/common.yaml
[PASS] fixed supervised manifests
  - train: {'COVID': 578, 'Lung_Opacity': 961, 'Viral_Pneumonia': 215, 'Normal': 1630} total=3384
  - val: {'COVID': 72, 'Lung_Opacity': 120, 'Viral_Pneumonia': 26, 'Normal': 203} total=421
  - test: {'COVID': 73, 'Lung_Opacity': 121, 'Viral_Pneumonia': 28, 'Normal': 205} total=427
[PASS] experiment config files
[PASS] resnet18_covidqu
  - planned output_dir: results/experiments/resnet18_covidqu
[PASS] resnet18_covidqu_syn
  - synthetic_dcgan: {'COVID': 1000, 'Lung_Opacity': 1000, 'Viral_Pneumonia': 1000, 'Normal': 1000} total=4000
  - planned output_dir: results/experiments/resnet18_covidqu_syn
[PASS] resnet18_imagenet
  - no contrastive pretraining data required
  - planned output_dir: results/experiments/resnet18_imagenet
[PASS] resnet18_imagenet_covidqu
  - planned output_dir: results/experiments/resnet18_imagenet_covidqu
[PASS] resnet18_imagenet_covidqu_

## 7. Resolve Experiment Settings

In [20]:
EXPERIMENTS = {
    'resnet18_covidqu': {
        'config': 'configs/experiments/resnet18/covidqu.yaml',
        'uses_synthetic': False,
    },
    'resnet18_imagenet_covidqu': {
        'config': 'configs/experiments/resnet18/imagenet_covidqu.yaml',
        'uses_synthetic': False,
    },
    'resnet18_covidqu_syn': {
        'config': 'configs/experiments/resnet18/covidqu_syn.yaml',
        'uses_synthetic': True,
    },
    'resnet18_imagenet_covidqu_syn': {
        'config': 'configs/experiments/resnet18/imagenet_covidqu_syn.yaml',
        'uses_synthetic': True,
    },
}

if EXPERIMENT_ID not in EXPERIMENTS:
    raise ValueError(f'Unsupported EXPERIMENT_ID: {EXPERIMENT_ID}')

EXP = EXPERIMENT_ID
CONFIG = EXPERIMENTS[EXP]['config']
USES_SYNTHETIC = EXPERIMENTS[EXP]['uses_synthetic']
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'

pretrain_epoch_arg = f'--epochs {PRETRAIN_EPOCHS}' if PRETRAIN_EPOCHS is not None else ''
finetune_epoch_arg = f'--epochs {FINETUNE_EPOCHS}' if FINETUNE_EPOCHS is not None else ''

print('EXP:', EXP)
print('CONFIG:', CONFIG)
print('OUT:', OUT)
print('CKPT:', CKPT)
print('RESUME_CKPT:', RESUME_CKPT)
print('USES_SYNTHETIC:', USES_SYNTHETIC)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

EXP: resnet18_covidqu_syn
CONFIG: configs/experiments/resnet18/covidqu_syn.yaml
OUT: /kaggle/working/results/experiments/resnet18_covidqu_syn
CKPT: /kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/checkpoints/best_simclr_backbone.pth
RESUME_CKPT: /kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth
USES_SYNTHETIC: True
pretrain_epoch_arg: --epochs 70
finetune_epoch_arg: 


## 8. Optional: Restore Previous Kaggle Result

If you uploaded a previous result folder as a Kaggle Dataset, this cell copies it into `/kaggle/working` before training so SimCLR can resume from `last_simclr_checkpoint.pth`.

In [21]:
if PREVIOUS_EXPERIMENT_SOURCE is not None:
    previous = Path(PREVIOUS_EXPERIMENT_SOURCE)
    if not previous.exists():
        raise FileNotFoundError(f'PREVIOUS_EXPERIMENT_SOURCE does not exist: {previous}')
    if OUT.exists():
        shutil.rmtree(OUT)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(previous, OUT)
    print('Restored previous result folder:', previous, '->', OUT)
else:
    print('No previous result folder configured. Starting from existing /kaggle/working output if present, otherwise fresh.')

print('Resume checkpoint exists:', RESUME_CKPT.exists(), RESUME_CKPT)


No previous result folder configured. Starting from existing /kaggle/working output if present, otherwise fresh.
Resume checkpoint exists: False /kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth


## 9. Run SimCLR Pretraining

This command resumes automatically from `last_simclr_checkpoint.pth` if it exists.

In [22]:
if USES_SYNTHETIC:
    !python scripts/run_simclr_resnet.py \
      --config "{CONFIG}" \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}
else:
    !python scripts/run_simclr_resnet.py \
      --config "{CONFIG}" \
      --real-unlabeled-dir data/processed/unlabelled_16934 \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {pretrain_epoch_arg}

Epoch 1/70 simclr_loss=4.4350
Epoch 2/70 simclr_loss=4.0471
Epoch 3/70 simclr_loss=3.8922
Epoch 4/70 simclr_loss=3.7422
Epoch 5/70 simclr_loss=3.6804
Epoch 6/70 simclr_loss=3.6380
Epoch 7/70 simclr_loss=3.5716
Epoch 8/70 simclr_loss=3.5403
Epoch 9/70 simclr_loss=3.5303
Epoch 10/70 simclr_loss=3.4990
Epoch 11/70 simclr_loss=3.4923
Epoch 12/70 simclr_loss=3.4720
Epoch 13/70 simclr_loss=3.4682
Epoch 14/70 simclr_loss=3.4578
Epoch 15/70 simclr_loss=3.4535
Epoch 16/70 simclr_loss=3.4226
Epoch 17/70 simclr_loss=3.4253
Epoch 18/70 simclr_loss=3.4038
Epoch 19/70 simclr_loss=3.3970
Epoch 20/70 simclr_loss=3.3834
Epoch 21/70 simclr_loss=3.3724
Epoch 22/70 simclr_loss=3.3815
Epoch 23/70 simclr_loss=3.3647
Epoch 24/70 simclr_loss=3.3759
Epoch 25/70 simclr_loss=3.3477
Epoch 26/70 simclr_loss=3.3561
Epoch 27/70 simclr_loss=3.3632
Epoch 28/70 simclr_loss=3.3442
Epoch 29/70 simclr_loss=3.3473
Epoch 30/70 simclr_loss=3.3333
Epoch 31/70 simclr_loss=3.3334
Epoch 32/70 simclr_loss=3.3214
Epoch 33/70 simcl

## 10. Run Supervised Fine-Tuning


In [23]:
import os, glob

img_dir = "/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images"
print(os.path.exists(img_dir))
print(len(glob.glob(img_dir + "/*.png")))
print(glob.glob(img_dir + "/*.png")[:5])

True
1000
['/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00479.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00134.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00619.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00854.png', '/kaggle/working/contrastive-synthesis-medcls_CVProject/data/processed/synthetic_dcgan/COVID/images/dcgan_COVID_00048.png']


In [24]:
!python scripts/run_classification_resnet.py \
  --config "{CONFIG}" \
  --manifest-dir data/manifests \
  --output-dir "{OUT}" \
  --pretrained-checkpoint "{CKPT}" \
  {finetune_epoch_arg}

Missing keys after SimCLR encoder load: ['fc.weight', 'fc.bias']
Epoch 1/70 train_loss=1.2785 val_loss=1.0488 val_acc=0.6366 val_f1_macro=0.5360
Epoch 2/70 train_loss=0.9315 val_loss=0.8457 val_acc=0.6936 val_f1_macro=0.5727
Epoch 3/70 train_loss=0.7967 val_loss=0.7585 val_acc=0.7126 val_f1_macro=0.6113
Epoch 4/70 train_loss=0.7244 val_loss=0.7040 val_acc=0.7387 val_f1_macro=0.6687
Epoch 5/70 train_loss=0.6699 val_loss=0.6656 val_acc=0.7506 val_f1_macro=0.6855
Epoch 6/70 train_loss=0.6265 val_loss=0.6338 val_acc=0.7482 val_f1_macro=0.6886
Epoch 7/70 train_loss=0.5908 val_loss=0.6068 val_acc=0.7625 val_f1_macro=0.7103
Epoch 8/70 train_loss=0.5696 val_loss=0.5857 val_acc=0.7743 val_f1_macro=0.7417
Epoch 9/70 train_loss=0.5378 val_loss=0.5645 val_acc=0.7767 val_f1_macro=0.7458
Epoch 10/70 train_loss=0.5101 val_loss=0.5478 val_acc=0.7886 val_f1_macro=0.7587
Epoch 11/70 train_loss=0.4864 val_loss=0.5403 val_acc=0.7933 val_f1_macro=0.7753
Epoch 12/70 train_loss=0.4635 val_loss=0.5190 val_acc

## 11. Display and Package Results

Download the zip from the Kaggle output panel, or create a Kaggle Dataset from `/kaggle/working/results` if you need to resume later.

In [25]:
import json

metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('WARNING: metrics.json not found:', metrics_path)

!find "{OUT}" -maxdepth 4 -type f | sort
!cd /kaggle/working && zip -qr "{EXP}_results.zip" results/experiments/"{EXP}"
print('Result zip:', Path('/kaggle/working') / f'{EXP}_results.zip')

,experiment_id,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_epoch,best_val_f1_macro
0,resnet18_covidqu_syn,0.875878,0.889557,0.872409,0.88059,0.876758,0.875878,0.876168,64,0.872572


/kaggle/working/results/experiments/resnet18_covidqu_syn/best_checkpoint.pth
/kaggle/working/results/experiments/resnet18_covidqu_syn/classification_report.csv
/kaggle/working/results/experiments/resnet18_covidqu_syn/config_resolved_simclr.yaml
/kaggle/working/results/experiments/resnet18_covidqu_syn/config_resolved.yaml
/kaggle/working/results/experiments/resnet18_covidqu_syn/confusion_matrix.png
/kaggle/working/results/experiments/resnet18_covidqu_syn/metrics.json
/kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/checkpoints/best_simclr_backbone.pth
/kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/checkpoints/last_simclr_checkpoint.pth
/kaggle/working/results/experiments/resnet18_covidqu_syn/pretrain/simclr_history.json
Result zip: /kaggle/working/resnet18_covidqu_syn_results.zip
